In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e5/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e5/train.csv
/kaggle/input/competitions/playground-series-s6e5/test.csv


In [2]:
# Load the Kaggle competition files.
# Keeping these paths centralized makes it easy to rerun in Kaggle without touching later cells.
DATA_DIR = "/kaggle/input/competitions/playground-series-s6e5"

df = pd.read_csv(f"{DATA_DIR}/train.csv")
TEST_PATH = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_submission = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 439140 entries, 0 to 439139
Data columns (total 16 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      439140 non-null  int64  
 1   Driver                  439140 non-null  object 
 2   Compound                439140 non-null  object 
 3   Race                    439140 non-null  object 
 4   Year                    439140 non-null  int64  
 5   PitStop                 439140 non-null  int64  
 6   LapNumber               439140 non-null  int64  
 7   Stint                   439140 non-null  int64  
 8   TyreLife                439140 non-null  float64
 9   Position                439140 non-null  int64  
 10  LapTime (s)             439140 non-null  float64
 11  LapTime_Delta           439140 non-null  float64
 12  Cumulative_Degradation  439140 non-null  float64
 13  RaceProgress            439140 non-null  float64
 14  Position_Change     

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import time


# Drop id
df = df.drop(columns=['id'])

# Prepare categorical columns
cat_cols = ['Driver', 'Compound', 'Race']
for col in cat_cols:
    df[col] = df[col].astype('category')

X = df.drop(columns=['PitNextLap'])
y = df['PitNextLap']

# Train-test split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"X_train shape: {X_train.shape}, X_val shape: {X_val.shape}")

# Initialize and train XGBoost
print("Training XGBoost baseline...")
start_time = time.time()
clf = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    enable_categorical=True,
    tree_method='hist',
    n_jobs=-1
)

clf.fit(X_train, y_train)
elapsed = time.time() - start_time
print(f"Training completed in {elapsed:.2f} seconds.")

# Evaluate
y_train_pred = clf.predict_proba(X_train)[:, 1]
y_val_pred = clf.predict_proba(X_val)[:, 1]

train_auc = roc_auc_score(y_train, y_train_pred)
val_auc = roc_auc_score(y_val, y_val_pred)

print(f"Train ROC-AUC: {train_auc:.5f}")
print(f"Validation ROC-AUC: {val_auc:.5f}")

X_train shape: (351312, 14), X_val shape: (87828, 14)
Training XGBoost baseline...
Training completed in 8.79 seconds.
Train ROC-AUC: 0.97477
Validation ROC-AUC: 0.93986


In [5]:
import pandas as pd
import numpy as np
import xgboost as xgb
import time


train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")
print("Data loaded. Train shape:", train_df.shape, "Test shape:", test_df.shape)
# Drop id from features but keep for output
train_ids = train_df['id']
test_ids = test_df['id']
train_df = train_df.drop(columns=['id'])
test_df = test_df.drop(columns=['id'])
# Feature engineering
print("Engineering features...")
for df in [train_df, test_df]:
    df['Estimated_Total_Laps'] = (df['LapNumber'] / (df['RaceProgress'] + 1e-8)).replace([np.inf, -np.inf], np.nan)
    df['Estimated_Total_Laps'] = df['Estimated_Total_Laps'].fillna(70).round().clip(10, 100)
    df['Remaining_Laps'] = df['Estimated_Total_Laps'] - df['LapNumber']
    df['Degradation_Rate'] = df['Cumulative_Degradation'] / (df['TyreLife'] + 1e-8)
    df['TyreLife_x_Degradation'] = df['TyreLife'] * df['Cumulative_Degradation']
# Median pit tyre life
pit_stops = train_df[train_df['PitNextLap'] == 1]
median_pit = pit_stops.groupby(['Race', 'Compound', 'Stint'])['TyreLife'].median().reset_index()
median_pit.rename(columns={'TyreLife': 'MedianPitTyreLife'}, inplace=True)
global_median = pit_stops['TyreLife'].median()
comp_median = pit_stops.groupby('Compound')['TyreLife'].median().to_dict()
train_df = train_df.merge(median_pit, on=['Race','Compound','Stint'], how='left')
train_df['MedianPitTyreLife'] = train_df['MedianPitTyreLife'].fillna(train_df['Compound'].map(comp_median)).fillna(global_median)
train_df['TyreLife_to_MedianPitRatio'] = train_df['TyreLife'] / (train_df['MedianPitTyreLife'] + 1e-8)
test_df = test_df.merge(median_pit, on=['Race','Compound','Stint'], how='left')
test_df['MedianPitTyreLife'] = test_df['MedianPitTyreLife'].fillna(test_df['Compound'].map(comp_median)).fillna(global_median)
test_df['TyreLife_to_MedianPitRatio'] = test_df['TyreLife'] / (test_df['MedianPitTyreLife'] + 1e-8)
# Median lap time
median_lap = train_df.groupby(['Race','Compound'])['LapTime (s)'].median().reset_index()
median_lap.rename(columns={'LapTime (s)': 'MedianLapTimeRC'}, inplace=True)
global_lap = train_df['LapTime (s)'].median()
train_df = train_df.merge(median_lap, on=['Race','Compound'], how='left')
train_df['MedianLapTimeRC'] = train_df['MedianLapTimeRC'].fillna(global_lap)
train_df['LapTime_Ratio'] = train_df['LapTime (s)'] / (train_df['MedianLapTimeRC'] + 1e-8)
test_df = test_df.merge(median_lap, on=['Race','Compound'], how='left')
test_df['MedianLapTimeRC'] = test_df['MedianLapTimeRC'].fillna(global_lap)
test_df['LapTime_Ratio'] = test_df['LapTime (s)'] / (test_df['MedianLapTimeRC'] + 1e-8)
# Categorical conversion
cat_cols = ['Driver','Compound','Race']
for col in cat_cols:
    train_df[col] = train_df[col].astype('category')
    test_df[col] = test_df[col].astype('category')
# Prepare features and target
y_train = train_df['PitNextLap']
X_train = train_df.drop(columns=['PitNextLap'])
X_test = test_df.copy()
print('Feature matrix shapes:', X_train.shape, X_test.shape)
# XGBoost model
clf = xgb.XGBClassifier(
    n_estimators=1000,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    gamma=0.1,
    min_child_weight=3,
    random_state=42,
    enable_categorical=True,
    tree_method='hist',
    n_jobs=-1,
    eval_metric='auc'
)
print('Training final model on full data...')
start = time.time()
clf.fit(X_train, y_train)
print(f'Training completed in {time.time() - start:.2f} seconds.')
print('Predicting on test set...')
test_pred = clf.predict_proba(X_test)[:,1]
pred_df = pd.DataFrame({'id': test_ids, 'PitNextLap': test_pred})


Data loaded. Train shape: (439140, 16) Test shape: (188165, 15)
Engineering features...
Feature matrix shapes: (439140, 22) (188165, 22)
Training final model on full data...
Training completed in 57.13 seconds.
Predicting on test set...


In [6]:
OUTPUT_PATH = "submission.csv"

# 1. Get probabilities
test_pred_prob = clf.predict_proba(X_test)[:, 1]

# 2. Convert probabilities to 0 or 1 using 0.5 threshold
test_pred_labels = (test_pred_prob >= 0.5).astype(int)

# 3. Create DataFrame with converted labels
pred_df = pd.DataFrame({
    'id': test_ids,
    'PitNextLap': test_pred_labels  # Ab yahan decimal ki jagah 0 ya 1 ayega
})

pred_df.to_csv(OUTPUT_PATH, index=False)

print(f'Predictions saved to {OUTPUT_PATH}')


Predictions saved to submission.csv
